<a href="https://colab.research.google.com/github/pratheeshsailor/INF-Data-EDA-/blob/main/Brew_INF_Audits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from urllib.parse import urlparse

file_path = "/content/US.xlsx"              #  change file name
df = pd.read_excel(file_path)

In [ ]:
pivot_table = pd.pivot_table(
    df[df["criteria_name"].str.startswith("amzn1", na=False)],
    index="mapper_id",
    columns="mapping_status",
    values="sku",
    aggfunc="count",
    # margins=True,
    fill_value=0
)
pivot_table

mapping_status,IF,INF,PIF,PINF,RIF,UNKNOWN
mapper_id,,,,,,
LLM_MISMATCH_azbmoham@amazon.com,4,0,0,0,0,0
LLM_MISMATCH_nkalyans@amazon.com,1,0,0,0,0,0
LLM_MISMATCH_rashriy@amazon.com,2,0,0,0,0,0
LLM_MISMATCH_smprashu@amazon.com,1,0,0,0,0,0
ablessil@amazon.com,3,47,0,0,0,15
afm_brew_automation,0,0,0,11,0,245
akansagg@amazon.com,17,486,11,0,0,26
akelneeh@amazon.com,9,501,0,0,3,281
akojsiri@amazon.com,3,313,0,0,0,148


In [ ]:
pivot_table = pivot_table.rename(
    index={"All": "Total"},
    columns={"All": "Total"}
)

In [ ]:
# Add IF + INF total column
pivot_table["IF_INF_Total"] = pivot_table.get("IF", 0) + pivot_table.get("INF", 0)

pivot_table


mapping_status,IF,INF,PIF,PINF,RIF,UNKNOWN,IF_INF_Total
mapper_id,,,,,,,
LLM_MISMATCH_azbmoham@amazon.com,4,0,0,0,0,0,4
LLM_MISMATCH_nkalyans@amazon.com,1,0,0,0,0,0,1
LLM_MISMATCH_rashriy@amazon.com,2,0,0,0,0,0,2
LLM_MISMATCH_smprashu@amazon.com,1,0,0,0,0,0,1
ablessil@amazon.com,3,47,0,0,0,15,50
afm_brew_automation,0,0,0,11,0,245,0
akansagg@amazon.com,17,486,11,0,0,26,503
akelneeh@amazon.com,9,501,0,0,3,281,510
akojsiri@amazon.com,3,313,0,0,0,148,316


In [ ]:
pivot_table['IF_Percentage'] = (
    (pivot_table['IF'] / pivot_table['IF_INF_Total']) * 100
).round(0)
pivot_table

mapping_status,IF,INF,PIF,PINF,RIF,UNKNOWN,IF_INF_Total,IF_Percentage
mapper_id,,,,,,,,
LLM_MISMATCH_azbmoham@amazon.com,4,0,0,0,0,0,4,100.0
LLM_MISMATCH_nkalyans@amazon.com,1,0,0,0,0,0,1,100.0
LLM_MISMATCH_rashriy@amazon.com,2,0,0,0,0,0,2,100.0
LLM_MISMATCH_smprashu@amazon.com,1,0,0,0,0,0,1,100.0
ablessil@amazon.com,3,47,0,0,0,15,50,6.0
afm_brew_automation,0,0,0,11,0,245,0,NaN
akansagg@amazon.com,17,486,11,0,0,26,503,3.0
akelneeh@amazon.com,9,501,0,0,3,281,510,2.0
akojsiri@amazon.com,3,313,0,0,0,148,316,1.0


In [ ]:
pivot_table = pivot_table.reset_index()

In [ ]:
final_df = pivot_table.copy()

# Remove @amazon.com
final_df["user"] = final_df["mapper_id"].str.replace("@amazon.com", "", regex=False)

# Empty columns (will be filled from EMP file)
final_df["Emp Name"] = ""
final_df["Manager"] = ""
final_df["Team Name"] = ""


In [ ]:
final_df = final_df.rename(columns={
    "IF_INF_Total": "Total Mappings",
    "IF": "No of IF's",
    "INF": "No of INF's",
    "IF_Percentage": "IF%"
})


In [ ]:
emp_df = pd.read_excel("/content/Emp.xlsx")

# Clean user column for safe matching
emp_df["user"] = emp_df["user"].astype(str).str.strip()
final_df["user"] = final_df["user"].astype(str).str.strip()


In [ ]:
final_df = final_df.merge(
    emp_df[["user", "Emp Name", "Manager", "Team Name"]],
    on="user",
    how="left"
)
final_df


,mapper_id,No of IF's,No of INF's,PIF,PINF,RIF,UNKNOWN,Total Mappings,IF%,user,Emp Name_x,Manager_x,Team Name_x,Emp Name_y,Manager_y,Team Name_y
0,LLM_MISMATCH_azbmoham@amazon.com,4,0,0,0,0,0,4,100.0,LLM_MISMATCH_azbmoham,,,,NaN,NaN,NaN
1,LLM_MISMATCH_nkalyans@amazon.com,1,0,0,0,0,0,1,100.0,LLM_MISMATCH_nkalyans,,,,NaN,NaN,NaN
2,LLM_MISMATCH_rashriy@amazon.com,2,0,0,0,0,0,2,100.0,LLM_MISMATCH_rashriy,,,,NaN,NaN,NaN
3,LLM_MISMATCH_smprashu@amazon.com,1,0,0,0,0,0,1,100.0,LLM_MISMATCH_smprashu,,,,NaN,NaN,NaN
4,ablessil@amazon.com,3,47,0,0,0,15,50,6.0,ablessil,,,,Blessing Alciya A,Saranya M L,US
5,afm_brew_automation,0,0,0,11,0,245,0,NaN,afm_brew_automation,,,,NaN,NaN,NaN
6,akansagg@amazon.com,17,486,11,0,0,26,503,3.0,akansagg,,,,AKANSHA AGGARWAL,Karthik S,US
7,akelneeh@amazon.com,9,501,0,0,3,281,510,2.0,akelneeh,,,,Neeharika Akella Venkata Sai,Karthik S,US
8,akojsiri@amazon.com,3,313,0,0,0,148,316,1.0,akojsiri,,,,Shirisha Akoju,Arun Sarathy S,US
9,anupriyo@amazon.com,4,233,0,0,1,60,237,2.0,anupriyo,,,,NaN,NaN,NaN


In [ ]:
final_df = final_df.rename(columns={
    "Emp Name_y": "Emp Name",
    "Manager_y": "Manager",
    "Team Name_y": "Team Name"
})
final_df = final_df.drop(
    columns=["Emp Name_x", "Manager_x", "Team Name_x"],
    errors="ignore"
)


In [ ]:
final_df["Total Audits"] = ""
final_df["Incremental IF'S"] = ""
final_df["Audited IF%"] = ""

In [ ]:
final_report = final_df[
    [
        "Emp Name",
        "user",
        "Manager",
        "Team Name",
        "Total Mappings",
        "No of IF's",
        "No of INF's",
        "IF%",
        "Total Audits",
        "Incremental IF'S",
        "Audited IF%"
    ]
]
final_report

,Emp Name,user,Manager,Team Name,Total Mappings,No of IF's,No of INF's,IF%,Total Audits,Incremental IF'S,Audited IF%
0,NaN,LLM_MISMATCH_azbmoham,NaN,NaN,4,4,0,100.0,,,
1,NaN,LLM_MISMATCH_nkalyans,NaN,NaN,1,1,0,100.0,,,
2,NaN,LLM_MISMATCH_rashriy,NaN,NaN,2,2,0,100.0,,,
3,NaN,LLM_MISMATCH_smprashu,NaN,NaN,1,1,0,100.0,,,
4,Blessing Alciya A,ablessil,Saranya M L,US,50,3,47,6.0,,,
5,NaN,afm_brew_automation,NaN,NaN,0,0,0,NaN,,,
6,AKANSHA AGGARWAL,akansagg,Karthik S,US,503,17,486,3.0,,,
7,Neeharika Akella Venkata Sai,akelneeh,Karthik S,US,510,9,501,2.0,,,
8,Shirisha Akoju,akojsiri,Arun Sarathy S,US,316,3,313,1.0,,,
9,NaN,anupriyo,NaN,NaN,237,4,233,2.0,,,


In [ ]:
output_path = "/content/final_report.xlsx"

final_report.to_excel(output_path, index=False)

from google.colab import files
files.download(output_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
low_eff_mappers = (
    pivot_table
    .reset_index()
    .query("IF_Percentage <= 10 & mapper_id != 'All'")
    ["mapper_id"]
    .tolist()
)
# low_eff_mappers = (
#     pivot_table
#     .reset_index()
#     .query("10 < IF_Percentage <= 20 & mapper_id != 'All'")
#     ["mapper_id"]
#     .tolist()
# )


In [ ]:
filtered_data = df[
    (df["criteria_name"].str.startswith("amzn1.goal", na=False)) &
    (df["mapper_id"].isin(low_eff_mappers)) &
    (df["mapping_status"] == "INF")
].copy()


In [ ]:
filtered_data["sku"] = (
   filtered_data["sku"]
    .astype(str)          # ensure string
    .str.zfill(10)        # pad with leading zeros
)

In [ ]:
output_path = "/content/INF_Data.xlsx"
filtered_data.to_excel(output_path, index=False)

from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
file_path = "/content/INF_Data.xlsx"
inf_df = pd.read_excel(file_path)

inf_df.head()

,import_time_epoch,namespace,sku,competitor_name,matching_type,qualifier,source,criteria_name,location,mapping_status,...,is_mkpl_comp,afm_mode_of_comp,criteria_limit,marketplaceid,event_date,ri_matcher_tags,suggestion_reason_code,suggestion_sub_reasons,matching_type_attributes,search_tool_attributes
0,1791901988073,AMZN,B07L93N5LV,SafeWay,EXACT,NEW,ONLINE,amzn1.goal.goalService.8d6b7fa8a2082415c1c3bf7...,Any,inf,...,False,NotEnabled,0,1,2026-10-13,[],NaN,NaN,NaN,NaN
1,1791911240097,AMZN,B0GG5XDTNL,Denniskirk_MCMT,EXACT,NEW,ONLINE,amzn1.goal.goalService.ea52379b889d55d820dbe17...,Any,inf,...,False,NotEnabled,0,1,2026-10-13,[],NaN,NaN,NaN,NaN
2,1791902012343,AMZN,B01MDV3UTV,CVS,EXACT,NEW,ONLINE,amzn1.goal.goalService.8d6b7fa8a2082415c1c3bf7...,Any,inf,...,False,NotEnabled,0,1,2026-10-13,[],NaN,NaN,NaN,NaN
3,1791911257894,AMZN,B0HL91Z418,Lenovo_MCMT,EXACT,NEW,ONLINE,amzn1.goal.goalService.ea52379b889d55d820dbe17...,Any,inf,...,False,IFOnly,0,1,2026-10-13,[],NaN,NaN,NaN,NaN
4,1791865016383,AMZN,B0H6NPZZP9,Lowes,EXACT,NEW,ONLINE,amzn1.goal.goalService.9c3ac00f0f2ec209266c0a9...,Any,inf,...,False,Regular,0,1,2026-10-13,[],NaN,NaN,NaN,NaN


In [ ]:
inf_df["sku_len"] = inf_df["sku"].str.len()
inf_df["sku_len"].value_counts()

,count
sku_len,
10,11357


In [ ]:
file_path = "/content/GL_list.xlsx"
GL_list = pd.read_excel(file_path)

GL_list.head()

,gl_id,gl_code
0,14,gl_book
1,15,gl_music
2,21,gl_toy
3,23,gl_electronics
4,60,gl_home_improvement


In [ ]:
file_path = "/content/mkpl_list.xlsx"
mkpl_list = pd.read_excel(file_path)

mkpl_list.head()

,mkpl_code,client_id,marketplaceid
0,GB,9,3
1,DE,10,4
2,FR,11,5
3,US,1,1
4,CA,13,7


In [ ]:
inf_df = pd.merge(inf_df, mkpl_list[['marketplaceid', 'client_id']], on='marketplaceid', how='left')
inf_df.head()

,import_time_epoch,namespace,sku,competitor_name,matching_type,qualifier,source,criteria_name,location,mapping_status,...,criteria_limit,marketplaceid,event_date,ri_matcher_tags,suggestion_reason_code,suggestion_sub_reasons,matching_type_attributes,search_tool_attributes,sku_len,client_id
0,1791901988073,AMZN,B07L93N5LV,SafeWay,EXACT,NEW,ONLINE,amzn1.goal.goalService.8d6b7fa8a2082415c1c3bf7...,Any,inf,...,0,1,2026-10-13,[],NaN,NaN,NaN,NaN,10,1
1,1791911240097,AMZN,B0GG5XDTNL,Denniskirk_MCMT,EXACT,NEW,ONLINE,amzn1.goal.goalService.ea52379b889d55d820dbe17...,Any,inf,...,0,1,2026-10-13,[],NaN,NaN,NaN,NaN,10,1
2,1791902012343,AMZN,B01MDV3UTV,CVS,EXACT,NEW,ONLINE,amzn1.goal.goalService.8d6b7fa8a2082415c1c3bf7...,Any,inf,...,0,1,2026-10-13,[],NaN,NaN,NaN,NaN,10,1
3,1791911257894,AMZN,B0HL91Z418,Lenovo_MCMT,EXACT,NEW,ONLINE,amzn1.goal.goalService.ea52379b889d55d820dbe17...,Any,inf,...,0,1,2026-10-13,[],NaN,NaN,NaN,NaN,10,1
4,1791865016383,AMZN,B0H6NPZZP9,Lowes,EXACT,NEW,ONLINE,amzn1.goal.goalService.9c3ac00f0f2ec209266c0a9...,Any,inf,...,0,1,2026-10-13,[],NaN,NaN,NaN,NaN,10,1


In [ ]:
inf_df.shape

(11357, 37)

In [ ]:
gl_map = (
    GL_list
    .drop_duplicates("gl_code")   # lookup table only
    .set_index("gl_code")["gl_id"]
)

inf_df["gl_id"] = inf_df["gl_code"].map(gl_map)

In [ ]:
inf_df.shape

(11357, 38)

In [ ]:
output_df = pd.DataFrame({
    "Namespace": "AMZN",
    "Sku": inf_df["sku"],
    "mapper_id": inf_df["mapper_id"],
    "Qualifier": "NEW",
    "CompId": inf_df["competitor_name"],
    "MarketplaceId": inf_df["marketplaceid"],
    "MatchingType": "EXACT",
    "Location": "",
    "Source": "ONLINE",
    "Tag": "BM_INF_AuditML_1410_US",
    "clientId": inf_df["client_id"],
    "compSearchUrl": "",
    "gl_code": inf_df["gl_code"],
    "gvBand": inf_df["band"],
    "gl_id": inf_df["gl_id"]
})


In [ ]:
import numpy as np

# fixed seed ensures reproducibility
np.random.seed(42)

output_df["rand"] = np.random.rand(len(inf_df))

output_df = output_df.sort_values(["mapper_id", "rand"])

output_df["group_rank"] = (
    output_df
    .groupby("mapper_id")
    .cumcount() + 1
)

In [ ]:
output_df.columns

Index(['Namespace', 'Sku', 'mapper_id', 'Qualifier', 'CompId', 'MarketplaceId',
       'MatchingType', 'Location', 'Source', 'Tag', 'clientId',
       'compSearchUrl', 'gl_code', 'gvBand', 'gl_id', 'rand', 'group_rank'],
      dtype='object')

In [ ]:
pivot_mapper_count = (
    output_df
    .pivot_table(
        index="mapper_id",
        values="Sku",   # or any non-null column
        aggfunc="count"
    )
    .reset_index()
    .rename(columns={"Sku": "mapper_count"})
)
pivot_mapper_count


,mapper_id,mapper_count
0,ablessil@amazon.com,47
1,akansagg@amazon.com,486
2,akelneeh@amazon.com,501
3,akojsiri@amazon.com,313
4,anupriyo@amazon.com,233
5,arisrasp@amazon.com,348
6,azbmoham@amazon.com,404
7,bihanifa@amazon.com,846
8,bolleraj@amazon.com,233
9,budhayas@amazon.com,189


In [ ]:
import pandas as pd

audit_dfs = []
audit_counts = []

for _, row in pivot_mapper_count.iterrows():
    mapper = row["mapper_id"]
    max_count = row["mapper_count"]

    while True:
        try:
            user_count = int(
                input(f"Enter audit count for {mapper} (max {max_count}): ")
            )
            if 0 <= user_count <= max_count:
                break
            else:
                print(f"❌ Enter a value between 0 and {max_count}")
        except ValueError:
            print("❌ Please enter a valid integer")

    # store entered count
    audit_counts.append({
        "mapper_id": mapper,
        "total_INF": max_count,
        "audit_count_entered": user_count
    })

    filtered_df = output_df[
        (output_df["mapper_id"] == mapper) &
        (output_df["group_rank"] <= user_count)
    ]

    audit_dfs.append(filtered_df)


Enter audit count for ablessil@amazon.com (max 47): 6
Enter audit count for akansagg@amazon.com (max 486): 57
Enter audit count for akelneeh@amazon.com (max 501): 58
Enter audit count for akojsiri@amazon.com (max 313): 37
Enter audit count for anupriyo@amazon.com (max 233): 27
Enter audit count for arisrasp@amazon.com (max 348): 41
Enter audit count for azbmoham@amazon.com (max 404): 48
Enter audit count for bihanifa@amazon.com (max 846): 100
Enter audit count for bolleraj@amazon.com (max 233): 27
Enter audit count for budhayas@amazon.com (max 189): 22
Enter audit count for chouanam@amazon.com (max 435): 51
Enter audit count for insachin@amazon.com (max 418): 49
Enter audit count for jawmdsaq@amazon.com (max 444): 52
Enter audit count for jidmat@amazon.com (max 600): 71
Enter audit count for jrahsehg@amazon.com (max 589): 69
Enter audit count for krrishch@amazon.com (max 112): 13
Enter audit count for magukris@amazon.com (max 394): 46
Enter audit count for mkupasna@amazon.com (max 921)

In [ ]:
final_audit_df = pd.concat(audit_dfs, ignore_index=True)
audit_count_df = pd.DataFrame(audit_counts)


In [ ]:
final_audit_df.mapper_id.unique()

array(['ablessil@amazon.com', 'akansagg@amazon.com',
       'akelneeh@amazon.com', 'akojsiri@amazon.com',
       'anupriyo@amazon.com', 'arisrasp@amazon.com',
       'azbmoham@amazon.com', 'bihanifa@amazon.com',
       'bolleraj@amazon.com', 'budhayas@amazon.com',
       'chouanam@amazon.com', 'insachin@amazon.com',
       'jawmdsaq@amazon.com', 'jidmat@amazon.com', 'jrahsehg@amazon.com',
       'krrishch@amazon.com', 'magukris@amazon.com',
       'mkupasna@amazon.com', 'mohandvi@amazon.com', 'nehakj@amazon.com',
       'prajunan@amazon.com', 'rashriy@amazon.com', 'renusasa@amazon.com',
       'saidsk@amazon.com', 'sanechak@amazon.com', 'smprashu@amazon.com',
       'summaybe@amazon.com', 'vaasuki@amazon.com', 'vramesho@amazon.com',
       'vsandemo@amazon.com'], dtype=object)

In [ ]:
final_audit_df = final_audit_df.drop(columns=["rand", "group_rank", "gl_code"])

In [ ]:
final_audit_df = final_audit_df.rename(columns={"gl_id": "gl"})

In [ ]:
desired_columns = [
    'Namespace', 'Sku', 'mapper_id', 'Qualifier', 'CompId', 'MarketplaceId',
    'MatchingType', 'Location', 'Source', 'Tag', 'clientId',
    'compSearchUrl', 'gl', 'gvBand'
]

final_audit_df = final_audit_df[desired_columns]

In [ ]:
final_audit_df.columns

Index(['Namespace', 'Sku', 'mapper_id', 'Qualifier', 'CompId', 'MarketplaceId',
       'MatchingType', 'Location', 'Source', 'Tag', 'clientId',
       'compSearchUrl', 'gl', 'gvBand'],
      dtype='object')

In [ ]:
final_audit_df.head()

,Namespace,Sku,mapper_id,Qualifier,CompId,MarketplaceId,MatchingType,Location,Source,Tag,clientId,compSearchUrl,gl,gvBand
0,AMZN,B0FD1TB55X,ablessil@amazon.com,NEW,Lowes,1,EXACT,,ONLINE,BM_INF_AuditML_1410_US,1,,469,D
1,AMZN,B0892T6JV2,ablessil@amazon.com,NEW,Lowes,1,EXACT,,ONLINE,BM_INF_AuditML_1410_US,1,,201,E
2,AMZN,B00002N6FF,ablessil@amazon.com,NEW,AcmeTools,1,EXACT,,ONLINE,BM_INF_AuditML_1410_US,1,,60,NaN
3,AMZN,B07F1KB74T,ablessil@amazon.com,NEW,BedBathBeyond,1,EXACT,,ONLINE,BM_INF_AuditML_1410_US,1,,201,C
4,AMZN,B0DB8WLRVL,ablessil@amazon.com,NEW,Belk,1,EXACT,,ONLINE,BM_INF_AuditML_1410_US,1,,79,D


In [ ]:
output_path = "/content/tag_file.xlsx"
final_audit_df.to_excel(output_path, index=False)

from google.colab import files
files.download(output_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
file_path = "/content/tag_file.xlsx"
tag_file = pd.read_excel(file_path)

tag_file.head()

,Namespace,Sku,mapper_id,Qualifier,CompId,MarketplaceId,MatchingType,Location,Source,Tag,clientId,compSearchUrl,gl,gvBand
0,AMZN,B0FD1TB55X,ablessil@amazon.com,NEW,Lowes,1,EXACT,NaN,ONLINE,BM_INF_AuditML_1410_US,1,NaN,469,D
1,AMZN,B0892T6JV2,ablessil@amazon.com,NEW,Lowes,1,EXACT,NaN,ONLINE,BM_INF_AuditML_1410_US,1,NaN,201,E
2,AMZN,B00002N6FF,ablessil@amazon.com,NEW,AcmeTools,1,EXACT,NaN,ONLINE,BM_INF_AuditML_1410_US,1,NaN,60,NaN
3,AMZN,B07F1KB74T,ablessil@amazon.com,NEW,BedBathBeyond,1,EXACT,NaN,ONLINE,BM_INF_AuditML_1410_US,1,NaN,201,C
4,AMZN,B0DB8WLRVL,ablessil@amazon.com,NEW,Belk,1,EXACT,NaN,ONLINE,BM_INF_AuditML_1410_US,1,NaN,79,D
